# Final Model (Research-Inspired)
This notebook builds a new PM2.5 forecasting model inspired by the GA-optimized LSTM research. We adapt the core ideas (time-aware features and careful validation) into a strong, reproducible regression pipeline for the provided Kaggle dataset.

## 1. Load Train/Test and Submission Templates
We load the provided CSV files and check basic structure.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


train_path = Path('/content/drive/MyDrive/kaggle/train.csv')
test_path = Path('/content/drive/MyDrive/kaggle/test.csv')
sub_path = Path('/content/drive/MyDrive/kaggle/sample_submission.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sub_df = pd.read_csv(sub_path)

print(train_df.head())
print(test_df.head())
print(sub_df.head())
print(train_df.dtypes)

   No      DEWP      TEMP      PRES       Iws        Is        Ir  \
0   1 -1.580878 -1.922250  0.443328 -0.441894 -0.069353 -0.137667   
1   2 -1.580878 -2.004228  0.345943 -0.379306 -0.069353 -0.137667   
2   3 -1.580878 -1.922250  0.248559 -0.343514 -0.069353 -0.137667   
3   4 -1.580878 -2.168183  0.248559 -0.280926 -0.069353 -0.137667   
4   5 -1.511594 -2.004228  0.151174 -0.218339 -0.069353 -0.137667   

              datetime   cbwd_NW   cbwd_SE   cbwd_cv  pm2.5  
0  2010-01-01 00:00:00  1.448138 -0.732019 -0.522096    NaN  
1  2010-01-01 01:00:00  1.448138 -0.732019 -0.522096    NaN  
2  2010-01-01 02:00:00  1.448138 -0.732019 -0.522096    NaN  
3  2010-01-01 03:00:00  1.448138 -0.732019 -0.522096    NaN  
4  2010-01-01 04:00:00  1.448138 -0.732019 -0.522096    NaN  
      No      DEWP      TEMP      PRES       Iws        Is        Ir  \
0  30677  1.190496  0.701029 -2.186052 -0.003982 -0.069353 -0.137667   
1  30678  1.121211  0.619051 -2.186052  0.031811 -0.069353 -0.137667 

## 2. Parse Datetime and Align Row IDs
We normalize timestamps so test rows line up with the submission template row IDs.

In [2]:
def to_row_id(dt_series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(dt_series)
    return (
        dt.dt.strftime("%Y-%m-%d ")
        + dt.dt.hour.astype(str)
        + ":"
        + dt.dt.minute.astype(str).str.zfill(2)
        + ":"
        + dt.dt.second.astype(str).str.zfill(2)
    )

for df in (train_df, test_df):
    df["datetime"] = pd.to_datetime(df["datetime"])

sub_df["row ID"] = sub_df["row ID"].astype(str)

test_df["row_id"] = to_row_id(test_df["datetime"])
train_df["row_id"] = to_row_id(train_df["datetime"])

missing_in_sub = ~test_df["row_id"].isin(sub_df["row ID"])
print("Test rows not found in sample_submission row IDs:", missing_in_sub.sum())
print("Sample row IDs not found in test rows:", (~sub_df["row ID"].isin(test_df["row_id"])).sum())

Test rows not found in sample_submission row IDs: 0
Sample row IDs not found in test rows: 0


## 3. Handle Missing Targets in Train
We count missing target values and impute them as in the referenced approach.

In [3]:
missing_before = train_df["pm2.5"].isna().sum()
print("Missing pm2.5 before:", missing_before)

train_df["pm2.5"] = train_df["pm2.5"].fillna(0.0)

missing_after = train_df["pm2.5"].isna().sum()
print("Missing pm2.5 after:", missing_after)

Missing pm2.5 before: 1921
Missing pm2.5 after: 0


## 4. Feature Engineering from Datetime
We create time-based and cyclic features while keeping the meteorological inputs.

In [4]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = df["datetime"]
    df["hour"] = dt.dt.hour
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["weekday"] = dt.dt.weekday
    df["dayofyear"] = dt.dt.dayofyear

    # Cyclic encodings
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)
    return df

train_fe = add_time_features(train_df)
test_fe = add_time_features(test_df)

# Drop identifiers not used for modeling
train_fe = train_fe.drop(columns=["No"], errors="ignore")
test_fe = test_fe.drop(columns=["No"], errors="ignore")

# Fill remaining missing values
train_fe = train_fe.fillna(0.0)
test_fe = test_fe.fillna(0.0)

train_fe.head()

,DEWP,TEMP,PRES,Iws,Is,Ir,datetime,cbwd_NW,cbwd_SE,cbwd_cv,...,row_id,hour,day,month,weekday,dayofyear,hour_sin,hour_cos,doy_sin,doy_cos
0,-1.580878,-1.922250,0.443328,-0.441894,-0.069353,-0.137667,2010-01-01 00:00:00,1.448138,-0.732019,-0.522096,...,2010-01-01 0:00:00,0,1,1,4,1,0.000000,1.000000,0.017213,0.999852
1,-1.580878,-2.004228,0.345943,-0.379306,-0.069353,-0.137667,2010-01-01 01:00:00,1.448138,-0.732019,-0.522096,...,2010-01-01 1:00:00,1,1,1,4,1,0.258819,0.965926,0.017213,0.999852
2,-1.580878,-1.922250,0.248559,-0.343514,-0.069353,-0.137667,2010-01-01 02:00:00,1.448138,-0.732019,-0.522096,...,2010-01-01 2:00:00,2,1,1,4,1,0.500000,0.866025,0.017213,0.999852
3,-1.580878,-2.168183,0.248559,-0.280926,-0.069353,-0.137667,2010-01-01 03:00:00,1.448138,-0.732019,-0.522096,...,2010-01-01 3:00:00,3,1,1,4,1,0.707107,0.707107,0.017213,0.999852
4,-1.511594,-2.004228,0.151174,-0.218339,-0.069353,-0.137667,2010-01-01 04:00:00,1.448138,-0.732019,-0.522096,...,2010-01-01 4:00:00,4,1,1,4,1,0.866025,0.500000,0.017213,0.999852


## 5. Feature/Target Split and Baseline Metrics
We build a chronological split and compute baseline RMSE/MAE.

In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

feature_cols = [c for c in train_fe.columns if c not in ["pm2.5", "datetime", "row_id"]]
X = train_fe[feature_cols]
y = train_fe["pm2.5"]

split_idx = int(len(train_fe) * 0.75)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]

baseline_pred = np.full_like(y_val, y_train.mean(), dtype=float)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))
baseline_mae = mean_absolute_error(y_val, baseline_pred)

print("Baseline RMSE:", baseline_rmse)
print("Baseline MAE:", baseline_mae)

Baseline RMSE: 102.24733478449858
Baseline MAE: 74.27160680963812


## 6. Model Training Pipeline (Regression)
We train a regression model with scaling and evaluate on the validation split.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

base_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

base_model.fit(X_train, y_train)
val_pred = base_model.predict(X_val)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_mae = mean_absolute_error(y_val, val_pred)

print("Validation RMSE:", val_rmse)
print("Validation MAE:", val_mae)

Validation RMSE: 80.92741754961787
Validation MAE: 54.44334159168949


## 7. Hyperparameter Search
We tune model parameters with a time-series aware split.

In [10]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
}

ts_cv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_dist,
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=ts_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

best_model = search.best_estimator_
val_pred_best = best_model.predict(X_val)
val_rmse_best = np.sqrt(mean_squared_error(y_val, val_pred_best))
val_mae_best = mean_absolute_error(y_val, val_pred_best)

print("Validation RMSE (best):", val_rmse_best)
print("Validation MAE (best):", val_mae_best)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 10}
Best CV RMSE: 82.84322390638293
Validation RMSE (best): 79.28943257471369
Validation MAE (best): 52.81811749930741


## 8. Train Final Model and Predict Test
Fit the tuned model on all training data and generate test predictions.

In [11]:
X_full = train_fe[feature_cols]
y_full = train_fe["pm2.5"]

best_model.fit(X_full, y_full)

X_test = test_fe[feature_cols]
test_pred = best_model.predict(X_test)
print("Test predictions shape:", test_pred.shape)

Test predictions shape: (13148,)


## 9. Build Submission with Sample Row IDs
Create a submission aligned to the sample submission row IDs and save it.

In [ ]:
def build_submission(preds, sample_path, row_ids, out_path):
    submission = pd.read_csv(sample_path)
    submission["row ID"] = submission["row ID"].astype(str)
    pred_series = pd.Series(preds, index=row_ids.astype(str))
    submission["pm2.5"] = submission["row ID"].map(pred_series)
    submission["pm2.5"] = submission["pm2.5"].fillna(float(np.nanmean(preds)))
    submission.to_csv(out_path, index=False)
    print("Saved submission to:", out_path)
    return submission

In [ ]:
# Submission will be built after the improved model predictions are created.

Saved submission to: submission.csv


## 10. Improved XGBoost Model
XGBoost with optimized hyperparameters and feature engineering. We avoid lag features since recursive prediction accumulates errors. Instead, we focus on time-based and weather features with careful tuning.

In [ ]:
%pip install xgboost lightgbm catboost -q

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Fresh data load
train_raw = pd.read_csv("train.csv")
test_raw = pd.read_csv("test.csv")

def create_enhanced_features(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # Time features
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['weekofyear'] = df['datetime'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['datetime'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    
    # Cyclical encodings
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # Weather interaction features
    if 'TEMP' in df.columns and 'DEWP' in df.columns:
        df['temp_dewp_diff'] = df['TEMP'] - df['DEWP']
    if 'PRES' in df.columns and 'TEMP' in df.columns:
        df['pres_temp_ratio'] = df['PRES'] / (df['TEMP'] + 1e-6)
    if 'Iws' in df.columns:
        df['wind_speed_sq'] = df['Iws'] ** 2
    
    return df

train_fe = create_enhanced_features(train_raw)
test_fe = create_enhanced_features(test_raw)

# Define feature columns
feature_cols = [c for c in train_fe.columns if c not in ['No', 'pm2.5', 'datetime', 'row_id']]
target_col = 'pm2.5'

print(f"Features ({len(feature_cols)}): {feature_cols}")

X_full = train_fe[feature_cols]
y_full = train_fe[target_col]

# Drop NaN targets
valid_mask = ~y_full.isna()
X_full = X_full[valid_mask]
y_full = y_full[valid_mask]
print(f"Training samples after dropping NaN: {len(X_full)}")

X_test = test_fe[feature_cols]

# Chronological split for validation
train_size = int(len(X_full) * 0.85)
X_tr, X_val = X_full.iloc[:train_size], X_full.iloc[train_size:]
y_tr, y_val = y_full.iloc[:train_size], y_full.iloc[train_size:]

print(f"Train: {len(X_tr)}, Val: {len(X_val)}")

# XGBoost with optimized params
print("\n--- Training XGBoost ---")
xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=100
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)

xgb_val_pred = xgb_model.predict(X_val)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_pred))
print(f"XGBoost Val RMSE: {xgb_rmse:.4f}")

# LightGBM
print("\n--- Training LightGBM ---")
lgb_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

lgb_val_pred = lgb_model.predict(X_val)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_val_pred))
print(f"LightGBM Val RMSE: {lgb_rmse:.4f}")

# CatBoost
print("\n--- Training CatBoost ---")
cat_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=200,
    early_stopping_rounds=100
)
cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

cat_val_pred = cat_model.predict(X_val)
cat_rmse = np.sqrt(mean_squared_error(y_val, cat_val_pred))
print(f"CatBoost Val RMSE: {cat_rmse:.4f}")

# Weighted ensemble based on inverse RMSE
weights = np.array([1/xgb_rmse, 1/lgb_rmse, 1/cat_rmse])
weights = weights / weights.sum()
print(f"\nEnsemble weights - XGB: {weights[0]:.3f}, LGB: {weights[1]:.3f}, CAT: {weights[2]:.3f}")

ensemble_val_pred = weights[0]*xgb_val_pred + weights[1]*lgb_val_pred + weights[2]*cat_val_pred
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
print(f"Ensemble Val RMSE: {ensemble_rmse:.4f}")

## 11. Retrain on Full Data & Generate Submission
Retrain all models on full training data and create ensemble predictions for submission.

In [ ]:
# Retrain all models on full training data
print("Retraining on full data...")

# XGBoost - full retrain
xgb_full = xgb.XGBRegressor(
    n_estimators=xgb_model.best_iteration if hasattr(xgb_model, 'best_iteration') else 1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=42
)
xgb_full.fit(X_full, y_full)

# LightGBM - full retrain
lgb_full = lgb.LGBMRegressor(
    n_estimators=lgb_model.best_iteration_ if hasattr(lgb_model, 'best_iteration_') else 1000,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_full.fit(X_full, y_full)

# CatBoost - full retrain
cat_full = CatBoostRegressor(
    iterations=cat_model.best_iteration_ if hasattr(cat_model, 'best_iteration_') else 1000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=0
)
cat_full.fit(X_full, y_full)

# Generate predictions
xgb_test_pred = xgb_full.predict(X_test)
lgb_test_pred = lgb_full.predict(X_test)
cat_test_pred = cat_full.predict(X_test)

# Ensemble predictions
final_preds = weights[0]*xgb_test_pred + weights[1]*lgb_test_pred + weights[2]*cat_test_pred
print(f"Final predictions shape: {final_preds.shape}")
print(f"Predictions range: [{final_preds.min():.2f}, {final_preds.max():.2f}]")

# Build submission
test_fe["row_id"] = to_row_id(test_fe["datetime"])
submission = build_submission(final_preds, sub_path, test_fe["row_id"], "submission.csv")
print("\nSubmission preview:")
submission.head(10)

## 12. Alternative: Optimized XGBoost with Hyperparameter Search
If the ensemble doesn't achieve target score, try this more aggressive hyperparameter search.

In [ ]:
# Skip this cell if ensemble already gives good results
# Uncomment and run if you need more aggressive tuning

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# More aggressive hyperparameter search
param_dist_xgb = {
    'n_estimators': [500, 1000, 1500, 2000],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'max_depth': [4, 5, 6, 7, 8],
    'min_child_weight': [5, 10, 15, 20],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 2.0],
}

ts_cv = TimeSeriesSplit(n_splits=5)

search_xgb = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42),
    param_distributions=param_dist_xgb,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=ts_cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Running hyperparameter search (this may take several minutes)...")
search_xgb.fit(X_full, y_full)

print(f"\nBest params: {search_xgb.best_params_}")
print(f"Best CV RMSE: {-search_xgb.best_score_:.4f}")

# Generate predictions with best model
best_xgb = search_xgb.best_estimator_
tuned_preds = best_xgb.predict(X_test)

# Save alternative submission
submission_tuned = build_submission(tuned_preds, sub_path, test_fe["row_id"], Path('/content/drive/MyDrive/kaggle/submission_tuned.csv'))
print("\nTuned submission saved to submission_tuned.csv")